# Save Explorer (Notebook Edition)

Navigate the EU5 save JSON tree interactively. This notebook wraps the CLI `explore.py` functions for point-and-click use — descend into keys, search, and resolve IDs without leaving Jupyter.

**Note:** The CLI version (`python -m toolbox.explore`) has a full REPL. This notebook gives you the same capabilities with the advantage of keeping multiple views open at once.

In [ ]:
import sys
from pathlib import Path

# Get the absolute path of the current directory
current_dir = Path.cwd()

# If 'notebooks' is in the path, move to its parent
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    # Assuming if we aren't in notebooks, we are already at root 
    # (or you can add logic to find the root)
    PROJECT_ROOT = current_dir

# Set the working directory
import os
os.chdir(PROJECT_ROOT)

# Add to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\PH\Documents\Claude\Projects\PDX Save Analyzer


In [ ]:
# === CONFIGURATION ===
SAVE_FILE   = "saves/autosave.eu5"
RAKALY_BIN  = "bin/rakaly/rakaly"
LOC_DIR     = "game-data/eu5/localization/english"

In [ ]:
from toolbox.save_loader import load_save
from toolbox.explore import _show_node, _navigate, _type_label, _resolve_label

save = load_save(SAVE_FILE, rakaly_bin=RAKALY_BIN, loc_dir=LOC_DIR, verbose=True)
data = save.raw
print("Save loaded.")

[save_loader] Parsing autosave.eu5 via rakaly...
[save_loader] Parsed OK.
[save_loader] Loading localisation from game-data\eu5\localization\english...


Save loaded.


[save_loader] Loaded 86,507 localisation entries.


## Save Summary

In [ ]:
from toolbox.explore import _show_info
_show_info(save)


════════════════════════════════════════════════════════════
  EU5 SAVE SUMMARY
════════════════════════════════════════════════════════════
  Date        : 1345.10.1
  Version     : 1.1.10
  Multiplayer : False
  Player      : cheap
  Country     : Greenland (GRL)
  Age         : Renaissance (age_2_renaissance)

  Country resources (currency_data):
    gold                      0.09814
    stability                 51.07478
    prestige                  0.865
    army_tradition            11.38642
    government_power          92.9655
    religious_influence       30.75
    purity                    60
    righteousness             90
    complacency               1.15858

  Total real countries: 2344
════════════════════════════════════════════════════════════



## Navigate

Use `show()` to view keys at the current path. Use `go("key")` to descend, `up()` to go back, and `goto("dot.path")` for absolute jumps.

In [ ]:
# Navigation helpers
_path_stack = [[]]

def _current():
    node = data
    for p in _path_stack[-1]:
        if isinstance(node, dict):
            node = node.get(p, node.get(str(p), {}))
        elif isinstance(node, list):
            node = node[int(p)]
    return node

def show(max_items=40):
    """Show keys at current path."""
    _show_node(save, _current(), _path_stack[-1], max_items=max_items)

def go(key):
    """Descend into a key."""
    node = _current()
    key_str = str(key)
    if isinstance(node, dict) and key_str in node:
        _path_stack.append(_path_stack[-1] + [key_str])
    elif isinstance(node, list):
        _path_stack.append(_path_stack[-1] + [key_str])
    else:
        print(f"Key {key!r} not found.")
        return
    show()

def up():
    """Go up one level."""
    if len(_path_stack) > 1:
        _path_stack.pop()
    show()

def goto(dotpath):
    """Jump to an absolute dot-separated path (e.g. 'countries.database.2186')."""
    parts = dotpath.split(".")
    node, ok = _navigate(data, parts)
    if ok:
        _path_stack.clear()
        _path_stack.append(parts)
        show()
    else:
        print(f"Path not found: {dotpath}")

def home():
    """Return to root."""
    _path_stack.clear()
    _path_stack.append([])
    show()

def find(term, max_results=30):
    """Search keys at current level for a substring."""
    node = _current()
    if not isinstance(node, dict):
        print("find only works on dict nodes.")
        return
    matches = [(k, v) for k, v in node.items() if term.lower() in k.lower()]
    if matches:
        print(f"\nMatches for {term!r} ({len(matches)} hits):")
        for k, v in matches[:max_results]:
            print(f"  {k:<40s}  {_type_label(v)}")
    else:
        print(f"No keys matching {term!r} at this level.")

def val():
    """Return the raw Python object at the current path (for further inspection)."""
    return _current()

print("Navigation ready. Functions: show(), go(key), up(), goto(dotpath), home(), find(term), val()")

Navigation ready. Functions: show(), go(key), up(), goto(dotpath), home(), find(term), val()


In [ ]:
# Start browsing from root
show()


📍 /
────────────────────────────────────────────────────────────
  metadata                                  dict  (11 keys)
  start_of_day                              str   '1345.10.1'
  current_age                               str   'age_2_renaissance'
  speed                                     int   6
  random_seed                               int   4135710060
  random_count                              int   8748099
  variables                                 dict  (2 keys)
  language_manager                          dict  (2 keys)
  ironman_manager                           dict  (2 keys)
  great_power_manager                       dict  (1 keys)
  road_network                              dict  (1 keys)
  resolution_manager                        dict  (1 keys)
  situation_manager                         dict  (22 keys)
  dynamic_game_object_manager               list  (0 × empty)
  institution_manager                       dict  (1 keys)
  loan_manager                      

In [ ]:
# Example: descend into countries
# go("countries")
# go("database")
# go("2186")       # a specific country
# up()              # back to database
# goto("countries.database.2186.currency_data")  # absolute jump
# home()            # back to root
goto("population") 



📍 /countries/database/2186
────────────────────────────────────────────────────────────
  country_name                              str   'WUR'
  flag                                      str   'WUR'
  definition                                str   'WUR'
  type                                      str   'location'
  historical                                str   'WUR'
  country_type                              str   'Real'
  score                                     dict  (3 keys)
  currency_data                             dict  (11 keys)
  balance_history_2                         dict  (11 keys)
  adult_capable_characters                  int   4
  possible_advisors                         int   4
  marriage                                  int   2
  estimated_monthly_income_trade_and_tax    float 8.392
  estimated_monthly_income                  float 7.957
  monthly_trade_balance                     float -0.4711
  monthly_trade_value                       float 0.7542
  curre